## A2.2 — Clasificación de los productos: línea priorizada y universo total

La A2.1 dejó una línea comercial priorizada (tabla `fs2_priorizacion_lineas`). En esta etapa el mismo método de clasificación se aplica a **dos alcances** y se obtienen dos resultados: (1) los productos clasificados de la línea priorizada, que es el insumo del modelo de pronóstico de la Fase 3, y (2) los productos clasificados del universo total de la Fase 1, que sirve para observar cómo actúa el clasificador con una muestra más grande y con líneas de comportamientos distintos. El pipeline no cambia entre un alcance y otro: solo cambia el conjunto de productos que entra, y todas las decisiones relativas (percentiles, winsorización, imputación, ABC) se calculan dentro de cada alcance.

La clasificación no persigue reducir el número de productos por sí misma. Su función es decidir **en cuáles productos un pronóstico más preciso aporta más valor y, al mismo tiempo, es técnicamente viable**, y qué tratamiento recibe el resto. Para ello cada producto se evalúa en dos ejes. El primero es la **prioridad**: importancia (valor y crecimiento) y costo actual del error en inventario (días en quiebre, cobertura, devoluciones y vencimientos). El segundo es la **pronosticabilidad**: regularidad de la demanda (variabilidad y meses atípicos), intermitencia (ADI y CV² mensuales de Syntetos, Boylan y Croston, 2005) y longitud de la historia. Los días en quiebre se usan solo en el primer eje, porque la demanda observada durante un quiebre está censurada y contarla también como indicador de pronosticabilidad la penalizaría dos veces.

Con pocos productos y sin etiquetas externas, la clasificación no puede apoyarse en un modelo supervisado: cualquier etiqueta saldría de las mismas reglas que se quiere evaluar. Por eso se aplican varias familias de métodos no supervisados o basados en reglas, se comparan en una matriz producto × método y se selecciona de allí el piloto por consenso, siguiendo la lógica de combinar métodos multicriterio y medir su acuerdo con correlaciones de rango (Kou et al., 2012) y la de agregar varias particiones en una sola (Strehl y Ghosh, 2002). La comparación de índices internos de clustering (silhouette, Davies-Bouldin, Calinski-Harabasz) sigue el esquema de Zietsman y van Vuuren (2022).

**Cómo está organizado el notebook.** Primero se definen los parámetros, los criterios, los métodos y las funciones de cada etapa. Luego se ejecutan las mismas etapas dos veces, una por alcance, y al final se comparan los dos resultados. Cada etapa deja sus resultados en un contexto (`cL` para la línea y `cT` para el total), de modo que pueden volver a correrse por separado.

> **Entradas y confidencialidad.** El notebook lee las tablas `gld_abt_sku` y `gld_abt_sku_mes` generadas en la Fase 1 y la tabla `fs2_priorizacion_lineas` de la actividad A2.1 (priorización de líneas comerciales). Esta versión se publica sin datos ni salidas, y las celdas de interpretación explican cómo leer cada resultado sin cifras del negocio.

In [ ]:
# ============================================================================
# FASE 2 — A2.2 a A4.1: clasificación de la línea priorizada y del universo total
# ============================================================================
import warnings; warnings.filterwarnings("ignore")
import time, textwrap
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from IPython.display import Markdown, display
from scipy.stats import spearmanr, rankdata, mannwhitneyu
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (silhouette_score, davies_bouldin_score, calinski_harabasz_score,
                             adjusted_rand_score, normalized_mutual_info_score)
from threadpoolctl import threadpool_limits
threadpool_limits(limits=1)   # evita la sobresuscripción de hilos (OpenMP/BLAS) en el driver de Spark
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110

# ---- Parámetros del método (los mismos para los dos alcances; se reportan como análisis de sensibilidad) ----
SEED           = 42
LINEA          = "NombreLinea"
CORTE_EJE      = 0.5      # un SKU es "alto" en un eje si su percentil de consenso supera este corte
VOTOS_MIN      = 0.5      # proporción mínima de métodos que deben proponer al SKU para el piloto
FREC_MIN       = 0.7      # proporción mínima de submuestras en que el SKU debe seguir siendo seleccionado
K_RANGO        = (3, 4, 5)
MIN_CLUSTER    = 3        # un cluster con menos SKUs no puede ser "el cluster piloto"
N_SUB, FRAC_SUB = 10, 0.8
N_SIM_PESOS    = 2000
ADI_CORTE, CV2_CORTE = 1.32, 0.49          # Syntetos, Boylan & Croston (2005)
UMBRAL_CENSURA = 10.0                      # % de días en quiebre desde el cual la demanda se considera censurada

# ---- Datos ----
sku_all = spark.table("gld_abt_sku").toPandas()
mes_all = spark.table("gld_abt_sku_mes").toPandas()
mes_all["mes"] = pd.to_datetime(mes_all["mes"])
prior = spark.table("fs2_priorizacion_lineas").toPandas()
LINEA_PRIORIZADA = prior.loc[prior["linea_priorizada"], LINEA].iloc[0]

# ---- Los dos alcances: el pipeline es el mismo, solo cambia el conjunto de productos ----
ALCANCES = {
    "linea": {"nombre": LINEA_PRIORIZADA, "piloto_max": 20,   # tope a validar con el asesor
              "tablas": {"cls": "fs2_clasificacion_linea", "abt": "fs2_abt_sku_piloto",
                         "mes_piloto": "fs2_abt_sku_mes_piloto", "mes_resto": "fs2_abt_sku_mes_resto_linea"}},
    "total": {"nombre": "Universo total", "piloto_max": None,           # sin tope: se observa cómo actúa el clasificador
              "tablas": {"cls": "fs2_clasificacion_total", "abt": "fs2_abt_sku_piloto_total",
                         "mes_piloto": "fs2_abt_sku_mes_piloto_total", "mes_resto": "fs2_abt_sku_mes_resto_total"}},
}

def nuevo_ctx(clave):
    cfg = ALCANCES[clave]
    if clave == "linea":
        s, m = sku_all[sku_all[LINEA] == LINEA_PRIORIZADA], mes_all[mes_all[LINEA] == LINEA_PRIORIZADA]
    else:
        s, m = sku_all, mes_all
    ctx = {"clave": clave, "nombre": cfg["nombre"], "piloto_max": cfg["piloto_max"], "tablas": cfg["tablas"],
           "sku": s.reset_index(drop=True).copy(), "mes": m.sort_values(["ItemCode", "mes"]).copy()}
    print(f"{ctx['nombre']} | SKUs: {len(ctx['sku'])} (de {len(sku_all)}) | filas mensuales: {len(ctx['mes']):,} | "
          f"meses: {ctx['mes']['mes'].min():%Y-%m} a {ctx['mes']['mes'].max():%Y-%m}")
    return ctx

# ---- Utilidad de gráficas: heatmap con TODAS las celdas anotadas ----
# (sns.heatmap(annot=True) deja celdas sin texto en algunas versiones de matplotlib; aquí el texto se escribe a mano)
def heat(M, ax, titulo, vmin=-1, vmax=1, cmap="RdBu_r", fmt=".2f", fs=7, cbar=False):
    sns.heatmap(M, cmap=cmap, vmin=vmin, vmax=vmax, annot=False, cbar=cbar, ax=ax, linewidths=.4, linecolor="white",
                cbar_kws={"shrink": .7} if cbar else None)
    cm = plt.get_cmap(cmap)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M.iloc[i, j]
            if pd.isna(v): continue
            c = cm((v - vmin) / (vmax - vmin)); lum = .299 * c[0] + .587 * c[1] + .114 * c[2]
            ax.text(j + .5, i + .5, format(v, fmt), ha="center", va="center", fontsize=fs,
                    color="white" if lum < .5 else "black")
    ax.set_title(titulo, fontsize=10)
    ax.tick_params(axis="x", labelrotation=90, labelsize=8); ax.tick_params(axis="y", labelrotation=0, labelsize=8)

print("Alcances:", {k: v["nombre"] for k, v in ALCANCES.items()})

### A2.2 (paso 1) — Criterios, valores faltantes y redundancia

La Fase 1 concluyó que varios criterios miden la misma dimensión y que se retendría uno por bloque redundante junto con los independientes. Ese criterio se aplica a cada eje: del bloque de tamaño se conserva el valor de 12 meses; del bloque de irregularidad, el CV de 12 meses; de la intermitencia, el ADI y el CV² mensuales, que además permiten replicar la clasificación de Syntetos-Boylan; y se mantienen los indicadores independientes (crecimiento, cobertura, devolución, vencimiento, meses atípicos e historia). Como los bloques de la Fase 1 se midieron sobre el universo completo, se vuelve a calcular la correlación de Spearman dentro de cada alcance y se señalan los pares con |ρ| ≥ 0,8 para decidir si alguno debe retirarse.

Los valores faltantes son pocos y se explican por la construcción de los indicadores, así que se tratan según su significado. Las tasas de devolución y de vencimiento no están definidas en productos sin facturación en los últimos 12 meses, y allí se asume cero porque no hubo eventos que medir. La cobertura no se define sin demanda, y se asigna el máximo del alcance por ser el caso más desfavorable. Los demás (por ejemplo, el crecimiento sin demanda en el semestre de referencia) se completan con la mediana del alcance. La columna `n_imputados` conserva cuántos criterios fueron imputados en cada producto para poder revisar después si esos productos entran al piloto.

In [ ]:
# Criterios de cada eje. Un criterio por bloque de redundancia (Fase 1, sección 3.3) más los independientes.
#   sentido = "beneficio": mayor valor => mayor puntaje del eje | "costo": mayor valor => menor puntaje
EJE_P = {   # PRIORIDAD: dónde un mejor pronóstico genera más valor
    "valor_12m":            ("beneficio", "Importancia"),
    "crecimiento_6m_pct":   ("beneficio", "Importancia"),
    "pct_dias_quiebre_12m": ("beneficio", "Costo del error"),
    "cobertura_meses":      ("beneficio", "Costo del error"),
    "tasa_devolucion_12m":  ("beneficio", "Costo del error"),
    "tasa_vencimiento_12m": ("beneficio", "Costo del error"),
}
EJE_F = {   # PRONOSTICABILIDAD: qué tan viable es pronosticar la demanda del SKU
    "cv_12m":                 ("costo",     "Regularidad"),
    "pct_meses_atipicos_12m": ("costo",     "Regularidad"),
    "adi_mes":                ("costo",     "Intermitencia"),
    "cv2_mes":                ("costo",     "Intermitencia"),
    "meses_historia":         ("beneficio", "Historia"),
}
CRIT = list(EJE_P) + list(EJE_F)
IMPUTAR_CERO = ["tasa_devolucion_12m", "tasa_vencimiento_12m"]   # sin facturación no hubo eventos que medir
IMPUTAR_MAX  = ["cobertura_meses"]                               # sin demanda, la cobertura es la peor posible

def etapa_preparar(ctx):
    """Valores faltantes: se cuantifican y se tratan según el significado de cada indicador."""
    sku = ctx["sku"]
    sku[CRIT] = sku[CRIT].replace([np.inf, -np.inf], np.nan)
    sku["n_imputados"] = sku[CRIT].isna().sum(axis=1)
    falt = sku[CRIT].isna().sum(); falt = falt[falt > 0]
    display(falt.rename("faltantes").reset_index().rename(columns={"index": "criterio"}) if len(falt) else "Sin faltantes en los criterios")
    sku[IMPUTAR_CERO] = sku[IMPUTAR_CERO].fillna(0)
    sku[IMPUTAR_MAX]  = sku[IMPUTAR_MAX].fillna(sku[IMPUTAR_MAX].max())
    sku[CRIT]         = sku[CRIT].fillna(sku[CRIT].median())         # resto: mediana del alcance
    ctx["n_imputados"] = int((sku["n_imputados"] > 0).sum())
    print(f"SKUs con algún criterio imputado: {ctx['n_imputados']} de {len(sku)}")

def etapa_redundancia(ctx):
    """Spearman entre criterios dentro del alcance: los bloques de la Fase 1 pueden no sostenerse en otro conjunto."""
    corr = ctx["sku"][CRIT].corr(method="spearman"); ctx["corr"] = corr
    pares = (corr.where(np.triu(np.ones(corr.shape, bool), 1)).stack().rename("rho").reset_index()
             .rename(columns={"level_0": "criterio_a", "level_1": "criterio_b"})
             .query("abs(rho) >= 0.8").sort_values("rho", key=abs, ascending=False))
    ctx["pares"] = pares
    print("Pares con |rho| >= 0.8 (considerar retirar uno de cada par):")
    display(pares if len(pares) else "ninguno")
    fig, ax = plt.subplots(figsize=(9.5, 8))
    heat(corr, ax, f"Correlación de Spearman entre criterios — {ctx['nombre']}", fs=7.5, cbar=True)
    plt.tight_layout(); plt.show()

### A2.2 (paso 2) — Métodos comparados

Todos los criterios se winsorizan entre los percentiles 5 y 95, porque el valor y la cobertura tienen colas muy largas y un solo extremo domina la normalización; luego se llevan a [0, 1] con el sentido alineado, de modo que 1 siempre signifique "mejor para el eje". Sobre esa matriz se aplican tres métodos multicriterio y tres esquemas de ponderación, lo que da nueve puntajes por eje y por producto:

| Familia | Método | Qué aporta |
|---|---|---|
| Multicriterio | TOPSIS | Cercanía a la solución ideal y lejanía de la peor; es el método ya usado en la A2.1. |
| Multicriterio | VIKOR | Compromiso entre el puntaje agregado y el peor criterio del producto (v = 0,5). |
| Multicriterio | PROMETHEE II | Flujo neto de preferencias por pares; no depende de una solución ideal. |
| Pesos | Iguales por dimensión, entropía, CRITIC | Del subjetivo al objetivo; CRITIC descuenta criterios correlacionados. |
| Clustering | K-means, Ward, GMM (covarianza diagonal) | Estructura natural de los productos, sin depender de pesos. |
| Reglas de referencia | ABC·SB, ABC·XYZ | Los métodos clásicos como línea base: clase A o B por valor, y patrón suave o errático (o CV < 1). |

Cada método termina en un voto binario: "este producto debería entrar al piloto". En los multicriterio, el voto es positivo cuando el producto supera el corte en **ambos** ejes; el voto de cada método es la mayoría de sus tres esquemas de pesos. En el clustering se vota por el cluster con mayor valor medio del mínimo entre los dos ejes, que es el de productos buenos en ambos, y se exige un tamaño mínimo de tres productos. El **consenso** de cada eje es el percentil promedio de los nueve puntajes, y la proporción de métodos que votan por un producto es su puntaje de consenso para el piloto.

In [ ]:
# ============================================================================
# Los métodos: normalización, pesos, MCDM (TOPSIS, VIKOR, PROMETHEE II), reglas y clustering
# ============================================================================
def normalizar(df, eje, p=(5, 95)):
    """Winsoriza (p5-p95), lleva a [0,1] y alinea el sentido: 1 = mejor para el eje."""
    cols = list(eje)
    X = df[cols].astype(float)
    X = X.clip(X.quantile(p[0] / 100), X.quantile(p[1] / 100), axis=1)
    R = (X - X.min()) / (X.max() - X.min()).replace(0, 1)
    for c in cols:
        if eje[c][0] == "costo":
            R[c] = 1 - R[c]
    return R

def _w(x):
    x = pd.Series(x, dtype=float).fillna(0).clip(lower=0)
    return x / x.sum() if x.sum() > 0 else pd.Series(1 / len(x), index=x.index)

def pesos(R, eje):
    """Tres esquemas: igual por dimensión, entropía y CRITIC (Diakoulaki et al., 1995)."""
    dims = pd.Series({c: eje[c][1] for c in R.columns})
    w_igual = (1 / dims.nunique()) / dims.map(dims.value_counts())
    P = (R + 1e-9) / (R + 1e-9).sum()
    E = -(P * np.log(P)).sum() / np.log(len(R))
    C = R.std(ddof=0) * (1 - R.corr().fillna(0)).sum()
    return pd.DataFrame({"igual": _w(w_igual), "entropia": _w(1 - E), "critic": _w(C)})

def topsis(R, w):
    V = R * w
    d_mas = np.sqrt(((V - V.max(0)) ** 2).sum(1)); d_men = np.sqrt(((V - V.min(0)) ** 2).sum(1))
    return d_men / (d_mas + d_men + 1e-12)

def vikor(R, w, v=0.5):
    f_mas, f_men = R.max(0), R.min(0)
    T = w * (f_mas - R) / np.where(f_mas - f_men == 0, 1, f_mas - f_men)
    S, Rm = T.sum(1), T.max(1)
    q = lambda a: (a - a.min()) / ((a.max() - a.min()) or 1)
    return 1 - (v * q(S) + (1 - v) * q(Rm))           # mayor = mejor

def promethee2(R, w):
    d = R[:, None, :] - R[None, :, :]
    pref = (np.clip(d, 0, 1) * w).sum(2)               # función de preferencia lineal con p = rango (=1)
    return (pref.sum(1) - pref.sum(0)) / (len(R) - 1)  # flujo neto

MCDM = {"TOPSIS": topsis, "VIKOR": vikor, "PROMETHEE": promethee2}

def puntuar_eje(df, eje):
    R = normalizar(df, eje); W = pesos(R, eje)
    S = pd.DataFrame({f"{m}|{wn}": f(R.values, W[wn].values) for m, f in MCDM.items() for wn in W.columns},
                     index=df.index)
    return S, R, W

# ---- Reglas de referencia (baselines): ABC por valor, XYZ por CV y Syntetos-Boylan ----
def reglas(df):
    v = df["valor_12m"].sort_values(ascending=False)
    sh = v / v.sum(); previo = sh.cumsum() - sh
    abc = pd.Series(np.where(previo < .80, "A", np.where(previo < .95, "B", "C")), index=v.index).reindex(df.index)
    xyz = pd.Series(np.select([df["cv_12m"] < .5, df["cv_12m"] < 1.0], ["X", "Y"], "Z"), index=df.index)
    a, c = df["adi_mes"], df["cv2_mes"]
    sb = pd.Series(np.select([(a < ADI_CORTE) & (c < CV2_CORTE), (a < ADI_CORTE) & (c >= CV2_CORTE),
                              (a >= ADI_CORTE) & (c < CV2_CORTE)], ["suave", "errática", "intermitente"], "lumpy"),
                   index=df.index)
    return pd.DataFrame({"abc": abc, "xyz": xyz, "sb": sb})

# ---- Clustering ----
ALGOS = {
    "KMeans": lambda k, s: KMeans(n_clusters=k, n_init=20, random_state=s),
    "Ward":   lambda k, s: AgglomerativeClustering(n_clusters=k, linkage="ward"),
    "GMM":    lambda k, s: GaussianMixture(n_components=k, covariance_type="diag", n_init=5, random_state=s),
}

def matriz_cluster(RP, RF):
    return StandardScaler().fit_transform(pd.concat([RP, RF], axis=1).values)

def validez_clusters(Z, ks=K_RANGO):
    filas = []
    for nombre, mk in ALGOS.items():
        for k in ks:
            lab = mk(k, SEED).fit_predict(Z)
            if len(set(lab)) < 2:
                continue
            filas.append({"algoritmo": nombre, "k": k, "silhouette": silhouette_score(Z, lab),
                          "davies_bouldin": davies_bouldin_score(Z, lab),
                          "calinski_harabasz": calinski_harabasz_score(Z, lab),
                          "tam_min": int(pd.Series(lab).value_counts().min())})
    return pd.DataFrame(filas)

# ---- Pipeline completo: de una tabla de SKUs a la matriz de votos ----
def correr_modelos(df, k):
    S_P, RP, WP = puntuar_eje(df, EJE_P)
    S_F, RF, WF = puntuar_eje(df, EJE_F)
    pP, pF = S_P.rank(pct=True), S_F.rank(pct=True)           # percentil dentro del alcance (1 = mejor)
    P_cons, F_cons = pP.mean(axis=1), pF.mean(axis=1)
    votos = pd.DataFrame(index=df.index)
    for m in MCDM:                                            # voto = mayoría de los 3 esquemas de pesos
        cols = [c for c in pP.columns if c.startswith(m + "|")]
        alto = (pP[cols] > CORTE_EJE) & (pF[cols] > CORTE_EJE)
        votos[m] = alto.mean(axis=1) >= 0.5
    Z = matriz_cluster(RP, RF); etiquetas = {}
    ajuste = np.minimum(P_cons, F_cons)                       # el cluster piloto es el de mayor "mínimo de ambos ejes"
    for nombre, mk in ALGOS.items():
        lab = pd.Series(mk(k, SEED).fit_predict(Z), index=df.index); etiquetas[nombre] = lab
        res = ajuste.groupby(lab).agg(["mean", "size"]).query("size >= @MIN_CLUSTER")
        votos[nombre] = lab == (res["mean"].idxmax() if len(res) else lab.value_counts().idxmax())
    rg = reglas(df)
    votos["ABC·SB"]  = rg["abc"].isin(["A", "B"]) & rg["sb"].isin(["suave", "errática"])
    votos["ABC·XYZ"] = rg["abc"].isin(["A", "B"]) & rg["xyz"].isin(["X", "Y"])
    return {"S_P": S_P, "S_F": S_F, "pP": pP, "pF": pF, "P": P_cons, "F": F_cons,
            "votos": votos, "etiquetas": etiquetas, "reglas": rg, "RP": RP, "RF": RF, "WP": WP, "WF": WF, "Z": Z}

FAMILIAS = {"Multicriterio": ["TOPSIS", "VIKOR", "PROMETHEE"], "Clustering": ["KMeans", "Ward", "GMM"],
            "Reglas": ["ABC·SB", "ABC·XYZ"]}
print("Métodos disponibles:", list(MCDM), list(ALGOS), "+ reglas ABC·SB y ABC·XYZ")

### Funciones de cada etapa

Las celdas siguientes solo definen funciones: no calculan nada hasta que se invocan en las secciones de resultados. Las gráficas de calor se dibujan con una función propia (`heat`) que escribe el valor en **todas** las celdas; la anotación automática de seaborn dejaba celdas sin texto en algunas versiones de matplotlib.

In [ ]:
# ============================================================================
# Etapas del análisis. Cada una recibe el contexto de un alcance (línea o total) y guarda sus resultados en él.
# ============================================================================
def etapa_k(ctx):
    """Calidad de los clusters y elección de k (criterios internos)."""
    sku = ctx["sku"]
    Z0 = matriz_cluster(normalizar(sku, EJE_P), normalizar(sku, EJE_F))
    val = validez_clusters(Z0); ctx["val"] = val
    display(val.round(3))
    ctx["k"] = int(val.groupby("k")["silhouette"].mean().idxmax())
    print(f"k elegido por silhouette promedio entre algoritmos: {ctx['k']}")
    nombres = {"silhouette": "Silhouette (mayor = mejor)", "davies_bouldin": "Davies-Bouldin (menor = mejor)",
               "calinski_harabasz": "Calinski-Harabasz (mayor = mejor)"}
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.9))
    for ax, (m, t) in zip(axes, nombres.items()):
        sns.lineplot(data=val, x="k", y=m, hue="algoritmo", style="algoritmo", markers=True, dashes=True, ax=ax)
        ax.axvline(ctx["k"], color="grey", ls=":", lw=1.2)
        ax.text(ctx["k"], ax.get_ylim()[1], f"k elegido = {ctx['k']} ", va="top", ha="right", fontsize=8, color="grey")
        ax.set_title(t, fontsize=10); ax.set_xlabel("Número de clusters (k)"); ax.set_ylabel(m.replace("_", " "))
        ax.set_xticks(list(K_RANGO)); ax.legend(title="", fontsize=8)
    fig.suptitle(f"Calidad interna de los clusters — {ctx['nombre']}", fontsize=11, y=1.03)
    plt.tight_layout(); plt.show()

def etapa_matriz(ctx, top=25):
    """La matriz SKU × método y el consenso."""
    sku = ctx["sku"]
    r = correr_modelos(sku, ctx["k"]); votos = r["votos"]
    cls = sku[["ItemCode", "ItemName", LINEA]].copy()
    cls = cls.join(r["reglas"]).join(votos.astype(int))
    cls["P_consenso"], cls["F_consenso"] = r["P"].round(3), r["F"].round(3)
    cls["votos_piloto"] = votos.mean(axis=1).round(3)
    cls["candidato"] = cls["votos_piloto"] >= VOTOS_MIN
    ctx.update({"r": r, "votos": votos, "cls": cls})
    print(f"Métodos en la matriz: {votos.shape[1]} | candidatos al piloto (>= {VOTOS_MIN:.0%} de votos): "
          f"{int(cls['candidato'].sum())} de {len(cls)}")
    display(cls.sort_values("votos_piloto", ascending=False).head(top))
    display(pd.concat({"Prioridad": r["WP"], "Pronosticabilidad": r["WF"]}).round(3).reset_index()
            .rename(columns={"level_0": "eje", "level_1": "criterio"}))

def kendall_w(S):
    """Coeficiente de concordancia de Kendall entre m rankings de n SKUs (sin corrección por empates)."""
    n, m = S.shape
    R = S.rank()
    return 12 * ((R.sum(axis=1) - R.sum(axis=1).mean()) ** 2).sum() / (m ** 2 * (n ** 3 - n))

def _bloque(M, a, b):
    sub = M.loc[FAMILIAS[a], FAMILIAS[b]].values
    if a == b:
        n = sub.shape[0]; return sub[np.triu_indices(n, 1)].mean() if n > 1 else np.nan
    return sub.mean()

def etapa_concordancia(ctx):
    """A4.1 (parte 1): concordancia entre métodos."""
    r, votos = ctx["r"], ctx["votos"]
    ctx["W_P"], ctx["W_F"] = kendall_w(r["S_P"]), kendall_w(r["S_F"])
    print(f"Kendall W — eje Prioridad: {ctx['W_P']:.3f} | eje Pronosticabilidad: {ctx['W_F']:.3f}   (1 = acuerdo total)")
    mets = list(votos.columns)
    ari = pd.DataFrame([[adjusted_rand_score(votos[a], votos[b]) for b in mets] for a in mets], index=mets, columns=mets)
    nmi = pd.DataFrame([[normalized_mutual_info_score(votos[a], votos[b]) for b in mets] for a in mets], index=mets, columns=mets)
    ctx["ari"], ctx["nmi"] = ari, nmi
    iu = np.triu_indices(len(mets), 1)
    ctx["ari_medio"], ctx["nmi_medio"] = ari.values[iu].mean(), nmi.values[iu].mean()
    fams = list(FAMILIAS)
    ctx["ari_fam"] = pd.DataFrame([[_bloque(ari, a, b) for b in fams] for a in fams], index=fams, columns=fams)
    fig, axes = plt.subplots(1, 3, figsize=(20, 6.2))
    cP, cF = r["S_P"].corr(method="spearman"), r["S_F"].corr(method="spearman")
    heat(cP, axes[0], "Spearman entre puntajes: eje Prioridad", vmin=min(0, np.floor(cP.values.min() * 10) / 10), vmax=1, cmap="viridis", fs=7.5)
    heat(cF, axes[1], "Spearman entre puntajes: eje Pronosticabilidad", vmin=min(0, np.floor(cF.values.min() * 10) / 10), vmax=1, cmap="viridis", fs=7.5)
    heat(ari, axes[2], "ARI entre decisiones piloto / no piloto", vmin=-0.2, vmax=1, cmap="viridis", fs=8)
    fig.suptitle(f"Concordancia entre métodos — {ctx['nombre']}", fontsize=12, y=1.0)
    plt.tight_layout(); plt.show()
    print(f"ARI promedio entre pares de métodos: {ctx['ari_medio']:.3f} | NMI promedio: {ctx['nmi_medio']:.3f}")
    print("ARI promedio por familia (diagonal = dentro de la familia; fuera = entre familias):")
    display(ctx["ari_fam"].round(2))

def etapa_estabilidad(ctx):
    """A4.1 (parte 2): pesos aleatorios y submuestreo del pipeline completo."""
    sku, r, cls, k = ctx["sku"], ctx["r"], ctx["cls"], ctx["k"]
    n = len(sku)
    # (a) Sensibilidad a los pesos (Dirichlet, como en la A2.1)
    rng = np.random.default_rng(SEED)
    RP, RF = r["RP"].values, r["RF"].values
    WPs = rng.dirichlet(np.ones(RP.shape[1]), N_SIM_PESOS); WFs = rng.dirichlet(np.ones(RF.shape[1]), N_SIM_PESOS)
    cnt = np.zeros(n)
    for wp, wf in zip(WPs, WFs):
        cnt += (rankdata(topsis(RP, wp)) / n > CORTE_EJE) & (rankdata(topsis(RF, wf)) / n > CORTE_EJE)
    cls["frec_pesos"] = (cnt / N_SIM_PESOS).round(3)
    # (b) Submuestreo: se repite TODO el pipeline con el 80 % de los SKUs
    rng = np.random.default_rng(SEED)
    veces, sel = pd.Series(0, index=sku.index), pd.Series(0, index=sku.index)
    ari_sub = {a: [] for a in ALGOS}
    t0 = time.time()
    for it in range(N_SUB):
        if it == 5:
            est = (time.time() - t0) / 5 * N_SUB
            print(f"Tiempo estimado total: {est / 60:.1f} min", "(si es demasiado, reduce N_SUB)" if est > 600 else "")
        if it and it % 25 == 0: print(f"  submuestra {it}/{N_SUB} | {(time.time() - t0) / 60:.1f} min")
        idx = np.sort(rng.choice(sku.index, size=int(n * FRAC_SUB), replace=False))
        rs = correr_modelos(sku.loc[idx], k)
        veces.loc[idx] += 1
        sel.loc[idx] += (rs["votos"].mean(axis=1) >= VOTOS_MIN).astype(int).values
        for a in ALGOS:
            ari_sub[a].append(adjusted_rand_score(r["etiquetas"][a].loc[idx], rs["etiquetas"][a]))
    cls["frec_submuestra"] = (sel / veces.replace(0, np.nan)).round(3)
    ctx["ari_sub"] = pd.DataFrame({a: pd.Series(v).describe()[["mean", "std", "min"]] for a, v in ari_sub.items()}).T
    print("ARI medio de los clusters bajo submuestreo (1 = idénticos a la muestra completa):")
    display(ctx["ari_sub"].round(3))
    display(cls[["frec_pesos", "frec_submuestra"]].describe().T.round(3))
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    for ax, col, t in zip(axes, ["frec_pesos", "frec_submuestra"],
                          ["Frecuencia ante pesos aleatorios", "Frecuencia ante submuestreo del 80 %"]):
        ax.hist(cls[col].dropna(), bins=np.linspace(0, 1, 11), color="#2a9d8f", edgecolor="white")
        ax.set_title(t, fontsize=10); ax.set_xlabel("Proporción de repeticiones en que el SKU es seleccionado")
        ax.set_ylabel("Número de SKUs")
        if col == "frec_submuestra":
            ax.axvline(FREC_MIN, color="k", ls="--"); ax.text(FREC_MIN, ax.get_ylim()[1], f" corte = {FREC_MIN:.0%}", va="top", fontsize=8)
    fig.suptitle(f"Estabilidad de la selección — {ctx['nombre']}", fontsize=11, y=1.03)
    plt.tight_layout(); plt.show()

In [ ]:
def etapa_emulador(ctx):
    """A4.1 (parte 3): emulador supervisado SOLO EXPLICATIVO (la etiqueta sale de los mismos criterios)."""
    from sklearn.tree import DecisionTreeClassifier, export_text
    from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
    sku, cls = ctx["sku"], ctx["cls"]
    Xe, ye = sku[CRIT], cls["candidato"].astype(int)
    ctx["emu_ba"] = None
    if ye.sum() >= 5 and (len(ye) - ye.sum()) >= 5:
        arbol = DecisionTreeClassifier(max_depth=3, min_samples_leaf=3, class_weight="balanced", random_state=SEED)
        cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=SEED)
        ba = cross_val_score(arbol, Xe, ye, cv=cv, scoring="balanced_accuracy")
        ctx["emu_ba"] = (ba.mean(), ba.std())
        print(f"Exactitud balanceada (CV 5x10): {ba.mean():.2f} ± {ba.std():.2f}   <- fidelidad al consenso, no validez")
        arbol.fit(Xe, ye)
        print(export_text(arbol, feature_names=CRIT))
        imp = pd.Series(arbol.feature_importances_, index=CRIT); imp = imp[imp > 0].sort_values()
        fig, ax = plt.subplots(figsize=(7, 0.45 * len(imp) + 1.2))
        ax.barh(imp.index, imp.values, color="#2a9d8f")
        for i, v in enumerate(imp.values): ax.text(v, i, f" {v:.2f}", va="center", fontsize=8)
        ax.set_xlabel("Importancia en el árbol (suma = 1)"); ax.set_title(f"Criterios que explican el consenso — {ctx['nombre']}", fontsize=10)
        plt.tight_layout(); plt.show()
    else:
        print("Muy pocos candidatos (o no candidatos) para entrenar el emulador; se omite.")

SEGMENTOS = ["1. Piloto (modelo de pronóstico)", "2. Prioritario y difícil (método robusto)",
             "3. Estable de bajo impacto (método simple)", "4. Bajo impacto y difícil (seguimiento)"]
SEG_COL = dict(zip(SEGMENTOS, ["#2a9d8f", "#e76f51", "#e9c46a", "#9aa5b1"]))

def segmento(f):
    if f["piloto"]:                                                   return SEGMENTOS[0]
    if f["P_consenso"] > CORTE_EJE and f["F_consenso"] <= CORTE_EJE:  return SEGMENTOS[1]
    if f["P_consenso"] <= CORTE_EJE and f["F_consenso"] > CORTE_EJE:  return SEGMENTOS[2]
    return SEGMENTOS[3]

def etapa_seleccion(ctx):
    """A2.3: selección del grupo piloto."""
    sku, cls, r = ctx["sku"], ctx["cls"], ctx["r"]
    cls["score_conjunto"] = (r["P"] * r["F"]).round(3)
    cls["censurado"] = sku["pct_dias_quiebre_12m"].values > UMBRAL_CENSURA
    cls["estable"] = cls["frec_submuestra"] >= FREC_MIN
    elegibles = cls[cls["candidato"] & cls["estable"]].sort_values("score_conjunto", ascending=False)
    tope = ctx["piloto_max"]
    elegidos = elegibles if tope is None else elegibles.head(tope)
    cls["piloto"] = cls.index.isin(elegidos.index)
    cls["segmento"] = cls.apply(segmento, axis=1)
    val_pil = sku.loc[cls["piloto"].values, "valor_12m"].sum() / sku["valor_12m"].sum()
    ctx["sel"] = {"candidatos": int(cls["candidato"].sum()), "estables": len(elegibles), "piloto": int(cls["piloto"].sum()),
                  "valor_piloto": val_pil, "tope": tope}
    print(f"Candidatos: {ctx['sel']['candidatos']} | candidatos estables: {len(elegibles)} | piloto final: {ctx['sel']['piloto']}"
          + (f" (tope {tope})" if tope else " (sin tope)"))
    print(f"Valor 12m del piloto: {val_pil:.1%} del valor de {ctx['nombre']}")
    seg = cls.assign(valor=sku["valor_12m"].values).groupby("segmento").agg(
        skus=("ItemCode", "size"), valor=("valor", "sum"), P=("P_consenso", "mean"), F=("F_consenso", "mean"),
        censurados=("censurado", "sum"))
    seg["pct_valor"] = seg.pop("valor") / sku["valor_12m"].sum() * 100
    display(seg.round(2).reset_index())
    ctx["seg_tabla"] = seg

    fig, axes = plt.subplots(1, 2, figsize=(15, 5.6), gridspec_kw={"width_ratios": [1.15, 1]})
    ax = axes[0]
    for s in SEGMENTOS:
        d = cls[cls["segmento"] == s]
        ax.scatter(d["F_consenso"], d["P_consenso"], s=48 if len(cls) < 100 else 26, color=SEG_COL[s],
                   alpha=.85, edgecolor="white", linewidth=.5, label=f"{s} (n={len(d)})")
    if len(cls) <= 60:
        for _, f in cls[cls["piloto"]].iterrows():
            ax.annotate(str(f["ItemCode"]), (f["F_consenso"], f["P_consenso"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
    ax.axvline(CORTE_EJE, ls="--", c="grey"); ax.axhline(CORTE_EJE, ls="--", c="grey")
    for x, y, t, ha, va in [(0.99, 0.99, "Alta prioridad\npronosticable", "right", "top"), (0.01, 0.99, "Alta prioridad\ndifícil", "left", "top"),
                            (0.99, 0.01, "Baja prioridad\npronosticable", "right", "bottom"), (0.01, 0.01, "Baja prioridad\ndifícil", "left", "bottom")]:
        ax.text(x, y, t, transform=ax.transAxes, ha=ha, va=va, fontsize=8, color="grey", style="italic")
    ax.set_xlim(-0.02, 1.06); ax.set_ylim(-0.02, 1.06)
    ax.set_xlabel("Pronosticabilidad (percentil de consenso; 1 = más pronosticable)"); ax.set_ylabel("Prioridad (percentil de consenso; 1 = más prioritario)")
    ax.set_title(f"Plano prioridad × pronosticabilidad — {ctx['nombre']}", fontsize=10); ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2, frameon=False)
    ax = axes[1]
    orden = cls.sort_values("frec_submuestra", ascending=False)
    ax.bar(range(len(orden)), orden["frec_submuestra"], color=np.where(orden["piloto"], "#2a9d8f", "#c3c9d0"), width=1.0, linewidth=0)
    ax.axhline(FREC_MIN, ls="--", c="k"); ax.text(len(orden) - 1, FREC_MIN + .01, f"corte = {FREC_MIN:.0%}", ha="right", fontsize=8)
    ax.set_xlabel("SKU (ordenado por frecuencia de selección)"); ax.set_ylabel("Frecuencia de selección en submuestras")
    ax.set_title("Estabilidad de la selección (verde = piloto final)", fontsize=10); ax.set_ylim(0, 1.05)
    plt.tight_layout(); plt.show()

# ---- Validez externa: backtest de origen móvil con métodos simples ----
H, N_ORIG, MIN_TRAIN, COL_Y = 1, 6, 9, "unidades_pedidas"
def f_naive(y): return y[-1]
def f_mm3(y):   return y[-3:].mean()
def f_ses(y, a=0.3):
    n = y[0]
    for v in y[1:]: n = a * v + (1 - a) * n
    return n
def f_sba(y, a=0.1):
    nz = np.flatnonzero(y)
    if len(nz) == 0: return 0.0
    z, p, q = y[nz[0]], nz[0] + 1, 1
    for t in range(nz[0] + 1, len(y)):
        if y[t] > 0: z, p, q = a * y[t] + (1 - a) * z, a * q + (1 - a) * p, 1
        else: q += 1
    return (1 - a / 2) * z / p
MET_PRON = {"naive": f_naive, "mm3": f_mm3, "ses": f_ses, "sba": f_sba}

def backtest(y):
    y = np.asarray(y, float); T = len(y)
    origenes = [o for o in range(T - N_ORIG - H + 1, T - H + 1) if o >= MIN_TRAIN]
    if len(origenes) < 3: return None
    esc = np.abs(np.diff(y[:origenes[0]])).mean()
    real = np.array([y[o + H - 1] for o in origenes])
    err = {m: np.array([abs(real[i] - f(y[:o])) for i, o in enumerate(origenes)]) for m, f in MET_PRON.items()}
    mase = {m: (e.mean() / esc if esc > 0 else np.nan) for m, e in err.items()}
    return {"mase_base": np.nanmean(list(mase.values())), "mase_min": np.nanmin(list(mase.values())),
            "err_base_sum": np.mean([e.sum() for e in err.values()]), "real_sum": real.sum(), "n_origenes": len(origenes)}

def _resumen(g):
    return pd.Series({"skus": len(g), "MASE_mediano": g["mase_base"].median(), "MASE_mejor_metodo": g["mase_min"].median(),
                      "WMAPE": g["err_base_sum"].sum() / g["real_sum"].sum()})

def etapa_validacion(ctx):
    """A2.3 / A4.1 (parte 4): ¿los SKUs 'pronosticables' se pronostican mejor?"""
    cls, mes = ctx["cls"], ctx["mes"]
    filas = []
    for cod, g in mes[~mes["mes_lanzamiento"].fillna(False).astype(bool)].groupby("ItemCode"):
        b = backtest(g.sort_values("mes")[COL_Y].values)
        if b: filas.append({"ItemCode": cod, **b})
    ctx["val_stats"] = None
    if not filas:
        print("Ningún SKU tiene historia suficiente para el backtest."); return
    bt = pd.DataFrame(filas)
    cls = cls.drop(columns=[c for c in bt.columns if c != "ItemCode" and c in cls.columns]).merge(bt, on="ItemCode", how="left")
    ctx["cls"] = cls
    ok = cls.dropna(subset=["mase_base"])
    print(f"SKUs con backtest: {len(ok)} de {len(cls)} | origen móvil: {N_ORIG} orígenes, horizonte {H} mes(es)")
    rho, p = spearmanr(ok["F_consenso"], ok["mase_base"])
    vs = {"n": len(ok), "rho": rho, "p": p, "med_piloto": np.nan, "med_resto": np.nan, "p_mw": np.nan}
    print(f"Spearman(pronosticabilidad, MASE) = {rho:.2f} (p={p:.3f})  -> se espera negativo")
    a, b = ok.loc[ok["piloto"], "mase_base"], ok.loc[~ok["piloto"], "mase_base"]
    if len(a) >= 3 and len(b) >= 3:
        vs.update(med_piloto=a.median(), med_resto=b.median(), p_mw=mannwhitneyu(a, b, alternative="less").pvalue)
        print(f"MASE mediano: piloto {a.median():.2f} vs resto {b.median():.2f} | Mann-Whitney (piloto < resto) p={vs['p_mw']:.3f}")
    ctx["val_stats"] = vs
    display(ok.groupby("segmento").apply(_resumen).round(3).reset_index())
    display(ok.groupby("censurado").apply(_resumen).round(3).reset_index())     # efecto de la censura por quiebres
    display(ok.groupby("sb").apply(_resumen).round(3).reset_index())            # contraste con Syntetos-Boylan

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    for s in SEGMENTOS:
        d = ok[ok["segmento"] == s]
        ax.scatter(d["F_consenso"], d["mase_base"], s=40 if len(ok) < 100 else 22, color=SEG_COL[s], alpha=.85, edgecolor="white", linewidth=.5, label=s)
    ax.axhline(1, ls=":", c="grey"); ax.text(0.99, 1, "MASE = 1: igual que el método ingenuo", fontsize=8, color="dimgrey", va="bottom", ha="right", transform=ax.get_yaxis_transform(), bbox=dict(facecolor="white", alpha=.8, edgecolor="none", pad=1.5))
    ax.set_xlabel("Pronosticabilidad (percentil de consenso)"); ax.set_ylabel("MASE medio de 4 métodos simples (menor = mejor)")
    ax.set_title(f"¿Se pronostican mejor los más pronosticables?  Spearman = {rho:.2f} (p = {p:.3f})", fontsize=10)
    ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2, frameon=False)
    ax = axes[1]
    orden = [s for s in SEGMENTOS if (ok["segmento"] == s).any()]
    datos = [ok.loc[ok["segmento"] == s, "mase_base"].values for s in orden]
    bp = ax.boxplot(datos, patch_artist=True, showfliers=True, medianprops={"color": "black"})
    for patch, s in zip(bp["boxes"], orden): patch.set_facecolor(SEG_COL[s]); patch.set_alpha(.85)
    ax.set_xticks(range(1, len(orden) + 1))
    ax.set_xticklabels([textwrap.fill(s.split(". ", 1)[1], 20) + f"\n(n={len(d)})" for s, d in zip(orden, datos)], fontsize=8)
    ax.axhline(1, ls=":", c="grey")
    ax.set_ylabel("MASE medio de 4 métodos simples"); ax.set_title("Error de pronóstico por segmento", fontsize=10)
    fig.suptitle(f"Validez externa — {ctx['nombre']}", fontsize=11, y=1.02)
    plt.tight_layout(); plt.show()

In [ ]:
def _nivel(x, cortes, etiquetas):
    for c, e in zip(cortes, etiquetas):
        if x < c: return e
    return etiquetas[-1]

def _desc_ari(x):
    return "coinciden casi por completo" if x >= .9 else "se parecen" if x >= .5 else "coinciden poco" if x >= .2 else "no coinciden"

def texto_resultados(ctx):
    """Redacta el análisis de resultados con las cifras de la ejecución (se actualiza si cambian los datos o los parámetros)."""
    sku, cls, val, k, sel = ctx["sku"], ctx["cls"], ctx["val"], ctx["k"], ctx["sel"]
    n = len(sku); vk = val[val["k"] == k]
    sil, db, tam = vk["silhouette"].mean(), vk["davies_bouldin"].mean(), int(vk["tam_min"].min())
    a = ctx["ari"]; fam = ctx["ari_fam"]
    cruce = np.nanmean([fam.loc[x, y] for x in FAMILIAS for y in FAMILIAS if x < y])
    nl = sku[LINEA].nunique()
    P = []
    ref = ctx["nombre"] if ctx["clave"] == "linea" else "todo el portafolio"
    P.append(f"**Datos.** Se clasificaron {n} productos " + (f"de {ctx['nombre']}" if ctx["clave"] == "linea" else f"del universo total ({nl} líneas comerciales)")
             + f". {ctx['n_imputados']} productos ({ctx['n_imputados'] / n:.0%}) tuvieron al menos un criterio imputado, y "
             f"{cls['censurado'].mean():.0%} registró más de {UMBRAL_CENSURA:.0f} % de días en quiebre, por lo que su demanda observada está censurada.")
    P.append(f"**Estructura.** El número de clusters elegido fue k = {k}, con un silhouette promedio de {sil:.2f} "
             f"({_nivel(sil, [.25, .5], ['estructura débil', 'estructura moderada', 'estructura fuerte'])}) y un Davies-Bouldin de {db:.2f}; "
             f"el cluster más pequeño tiene {tam} productos. K-means y Ward {_desc_ari(a.loc['KMeans', 'Ward'])} (ARI = {a.loc['KMeans', 'Ward']:.2f}), "
             f"y K-means y GMM {_desc_ari(a.loc['GMM', 'KMeans'])} (ARI = {a.loc['GMM', 'KMeans']:.2f}) en cuanto al grupo que proponen para el piloto.")
    P.append(f"**Concordancia.** La W de Kendall fue {ctx['W_P']:.2f} en el eje de prioridad "
             f"({_nivel(ctx['W_P'], [.5, .7], ['acuerdo débil', 'acuerdo moderado', 'acuerdo fuerte'])}) y {ctx['W_F']:.2f} en el de pronosticabilidad "
             f"({_nivel(ctx['W_F'], [.5, .7], ['acuerdo débil', 'acuerdo moderado', 'acuerdo fuerte'])}). "
             f"Al comparar las decisiones piloto / no piloto, el ARI promedio dentro de los métodos multicriterio fue {fam.loc['Multicriterio', 'Multicriterio']:.2f}, "
             f"dentro del clustering {fam.loc['Clustering', 'Clustering']:.2f} y entre las dos reglas clásicas {fam.loc['Reglas', 'Reglas']:.2f}; "
             f"entre familias el promedio fue {cruce:.2f}"
             + (", lo que indica que cada familia identifica grupos distintos y que el consenso por votos aporta más que cualquier método aislado." if cruce < .3
                else ", lo que indica un acuerdo parcial entre familias." if cruce < .6 else ", lo que indica que las familias llegan a decisiones muy parecidas.")
             + f" Entre las reglas clásicas y el multicriterio el ARI fue {fam.loc['Multicriterio', 'Reglas']:.2f}"
             + (", es decir, las reglas seleccionan productos distintos a los que se priorizan al combinar valor, riesgo y pronosticabilidad." if fam.loc['Multicriterio', 'Reglas'] < .3
                else ", es decir, las reglas clásicas recuperan parte de lo que identifica el multicriterio."))
    tope = ("" if not sel["tope"] else f", y el tope de {sel['tope']} productos recortó el grupo" if sel["estables"] > sel["tope"] else f", y el tope de {sel['tope']} productos no fue limitante")
    seg = ctx["seg_tabla"]
    P.append(f"**Selección.** {sel['candidatos']} productos ({sel['candidatos'] / n:.0%}) fueron candidatos al reunir al menos {VOTOS_MIN:.0%} de los votos; "
             f"{sel['estables']} de ellos fueron estables en el submuestreo (frecuencia de al menos {FREC_MIN:.0%}){tope}. "
             f"El grupo piloto final tiene {sel['piloto']} productos y concentra el {sel['valor_piloto']:.1%} del valor a 12 meses de {ref}. "
             "Distribución por segmento: " + "; ".join(f"{s.split('. ', 1)[1].split(' (')[0].lower()}: {int(seg.loc[s, 'skus'])}" for s in SEGMENTOS if s in seg.index) + ".")
    fs = cls["frec_submuestra"]
    P.append(f"**Estabilidad.** La frecuencia de selección mediana fue {fs.median():.2f}, y {(fs >= FREC_MIN).mean():.0%} de los productos superó el corte de {FREC_MIN:.0%}. "
             f"Bajo submuestreo del 80 %, el ARI medio de los clusters respecto de la muestra completa fue "
             + ", ".join(f"{x} {ctx['ari_sub'].loc[x, 'mean']:.2f}" for x in ALGOS) + f" ({_nivel(ctx['ari_sub']['mean'].mean(), [.5, .8], ['particiones poco estables', 'particiones moderadamente estables', 'particiones estables'])}).")
    vs = ctx["val_stats"]
    if vs:
        if vs["p"] < .05:
            rel = "negativa y significativa, como se esperaba" if vs["rho"] < 0 else "positiva y significativa, contraria a lo esperado"
        else:
            rel = ("sin relación estadísticamente significativa (el signo negativo va en la dirección esperada)" if vs["rho"] < 0
                   else "sin relación estadísticamente significativa (el signo positivo es contrario a lo esperado, aunque no significativo)")
        P.append(f"**Validez externa.** En el backtest con {vs['n']} productos, la correlación de Spearman entre pronosticabilidad y MASE fue {vs['rho']:.2f} (p = {vs['p']:.3f}): " + rel + "."
                 + (f" El MASE mediano del piloto ({vs['med_piloto']:.2f}) " + ("fue menor" if vs["med_piloto"] < vs["med_resto"] else "no fue menor")
                    + f" que el del resto ({vs['med_resto']:.2f}; Mann-Whitney p = {vs['p_mw']:.3f})." if not np.isnan(vs["med_piloto"]) else ""))
    if ctx.get("emu_ba"):
        P.append(f"**Emulador.** Un árbol de profundidad 3 reproduce el consenso con una exactitud balanceada de {ctx['emu_ba'][0]:.2f} ± {ctx['emu_ba'][1]:.2f}; "
                 "esta cifra mide la fidelidad a la regla de consenso, no su validez.")
    caut = []
    if n < 60: caut.append(f"con {n} productos los índices de calidad de clusters y las pruebas tienen poca potencia")
    if tam < 2 * MIN_CLUSTER: caut.append(f"el cluster más pequeño ({tam} productos) es casi del tamaño mínimo exigido")
    if vs and vs["p"] >= .05: caut.append("la validez externa no alcanza significancia estadística")
    if sel["tope"] and sel["estables"] > sel["tope"]: caut.append(f"el tope de {sel['tope']} dejó fuera a {sel['estables'] - sel['tope']} productos elegibles")
    if caut: P.append("**Cautelas.** " + "; ".join(caut)[0].upper() + "; ".join(caut)[1:] + ".")
    ctx["texto"] = "\n\n".join(P)
    display(Markdown(ctx["texto"]))

In [ ]:
def sensibilidad_umbrales(ctx):
    """Número de candidatos según el corte por eje (CORTE_EJE) y la proporción mínima de votos (VOTOS_MIN)."""
    global CORTE_EJE
    original, filas = CORTE_EJE, {}
    try:
        for ce in (0.4, 0.5, 0.6):
            CORTE_EJE = ce
            share = correr_modelos(ctx["sku"], ctx["k"])["votos"].mean(axis=1)
            filas[f"CORTE_EJE = {ce}"] = {f"VOTOS_MIN = {vm}": int((share >= vm).sum()) for vm in (0.375, 0.5, 0.625)}
    finally:
        CORTE_EJE = original
    t = pd.DataFrame(filas).T
    ctx["sens_umbrales"] = t
    print(f"Candidatos según los umbrales — {ctx['nombre']} ({len(ctx['sku'])} productos; 8 métodos: 0,375 = 3 votos, 0,5 = 4, 0,625 = 5):")
    display(t)

# ---- Los 'Markdown' que acompañan cada etapa están en las celdas de texto; aquí solo se ejecuta ----
print("Funciones de etapa listas.")

In [ ]:
# ============================================================================
# A2.4 — Filtrado de la ABT y persistencia (un juego de tablas por alcance)
# ============================================================================
import re
def _a_spark(pdf):
    pdf = pdf.reset_index(drop=True).copy()
    pdf.columns = [re.sub(r"[^0-9A-Za-z_]+", "_", c).strip("_") for c in pdf.columns]   # nombres válidos para Delta (p. ej. ABC·SB)
    for c in pdf.columns:
        if pdf[c].dtype == "object": pdf[c] = pdf[c].astype("string")
    return spark.createDataFrame(pdf)

def guardar(pdf, nombre):
    _a_spark(pdf).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(nombre)
    print(f"Guardada {nombre}: {len(pdf):,} filas x {pdf.shape[1]} columnas")

def etapa_guardar(ctx):
    cls, T = ctx["cls"], ctx["tablas"]
    cols = ["ItemCode", "ItemName", LINEA, "abc", "xyz", "sb", "P_consenso", "F_consenso", "votos_piloto", "frec_pesos",
            "frec_submuestra", "candidato", "censurado", "segmento", "piloto", "mase_base", "mase_min"] + list(ctx["votos"].columns)
    piloto_ids = set(cls.loc[cls["piloto"], "ItemCode"]); resto_ids = set(cls["ItemCode"]) - piloto_ids
    guardar(cls[[c for c in cols if c in cls.columns]], T["cls"])
    guardar(sku_all[sku_all["ItemCode"].isin(piloto_ids)], T["abt"])
    guardar(mes_all[mes_all["ItemCode"].isin(piloto_ids)], T["mes_piloto"])
    guardar(mes_all[mes_all["ItemCode"].isin(resto_ids)], T["mes_resto"])
    return piloto_ids

In [ ]:
# ============================================================================
# Comparación de los dos alcances: ¿cómo actúa el clasificador al pasar de la línea al universo total?
# ============================================================================
from sklearn.metrics import cohen_kappa_score

def comparar_alcances(cL, cT):
    def fila(c):
        vk = c["val"][c["val"]["k"] == c["k"]]; fam = c["ari_fam"]; vs = c["val_stats"] or {}
        fs = c["cls"]["frec_submuestra"]
        return {"Productos": len(c["sku"]), "k elegido": c["k"], "Silhouette medio (k elegido)": vk["silhouette"].mean(),
                "Kendall W prioridad": c["W_P"], "Kendall W pronosticabilidad": c["W_F"], "ARI medio entre métodos": c["ari_medio"],
                "ARI entre familias (promedio)": np.nanmean([fam.loc[x, y] for x in FAMILIAS for y in FAMILIAS if x < y]),
                "Candidatos": c["sel"]["candidatos"], "% candidatos": c["sel"]["candidatos"] / len(c["sku"]) * 100,
                "Productos en el piloto": c["sel"]["piloto"], "% del valor en el piloto": c["sel"]["valor_piloto"] * 100,
                "Frecuencia de selección mediana": fs.median(), "% de SKUs con frecuencia >= corte": (fs >= FREC_MIN).mean() * 100,
                "Exactitud del emulador": c["emu_ba"][0] if c.get("emu_ba") else np.nan,
                "Spearman pronosticabilidad-MASE": vs.get("rho", np.nan)}
    tabla = pd.DataFrame({cL["nombre"]: fila(cL), cT["nombre"]: fila(cT)})
    display(tabla.round(2))
    return tabla

def consistencia_linea_total(cL, cT):
    """Los SKUs de la línea priorizada clasificados dos veces: solos y dentro del universo total."""
    cols = ["ItemCode", "P_consenso", "F_consenso", "votos_piloto", "candidato", "piloto"]
    m = cL["cls"][cols].merge(cT["cls"][cols], on="ItemCode", suffixes=("_linea", "_total"))
    rP, _ = spearmanr(m["P_consenso_linea"], m["P_consenso_total"]); rF, _ = spearmanr(m["F_consenso_linea"], m["F_consenso_total"])
    kappa = cohen_kappa_score(m["candidato_linea"], m["candidato_total"])
    cruz = pd.crosstab(m["candidato_linea"].map({True: "Candidato en la línea", False: "No candidato en la línea"}),
                       m["candidato_total"].map({True: "Candidato en el total", False: "No candidato en el total"}))
    display(cruz)
    print(f"Spearman de los percentiles de los {len(m)} SKUs de la línea: prioridad {rP:.2f} | pronosticabilidad {rF:.2f} | kappa de candidatos = {kappa:.2f}")
    cambian = m[m["candidato_linea"] != m["candidato_total"]].merge(cL["cls"][["ItemCode", "ItemName"]], on="ItemCode")
    if len(cambian):
        print("Productos cuyo estado de candidato cambia entre alcances:")
        display(cambian[["ItemCode", "ItemName", "P_consenso_linea", "P_consenso_total", "F_consenso_linea", "F_consenso_total",
                         "votos_piloto_linea", "votos_piloto_total"]].round(2))
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
    for ax, e, rr, t in zip(axes, ["P", "F"], [rP, rF], ["Prioridad", "Pronosticabilidad"]):
        col = np.where(m["candidato_linea"] & m["candidato_total"], "#2a9d8f", np.where(m["candidato_linea"] | m["candidato_total"], "#e76f51", "#c3c9d0"))
        ax.scatter(m[f"{e}_consenso_linea"], m[f"{e}_consenso_total"], c=col, s=45, edgecolor="white")
        ax.plot([0, 1], [0, 1], ls="--", c="grey", lw=1)
        ax.set_xlabel(f"Percentil de {t.lower()} dentro de la línea"); ax.set_ylabel(f"Percentil de {t.lower()} dentro del universo total")
        ax.set_title(f"{t}: Spearman = {rr:.2f}", fontsize=10); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    axes[0].scatter([], [], c="#2a9d8f", label="Candidato en ambos"); axes[0].scatter([], [], c="#e76f51", label="Candidato solo en uno")
    axes[0].scatter([], [], c="#c3c9d0", label="No candidato en ninguno"); axes[0].legend(fontsize=8, loc="upper left")
    fig.suptitle(f"Los SKUs de {cL['nombre']} clasificados solos y dentro del universo total", fontsize=11, y=1.02)
    plt.tight_layout(); plt.show()
    return {"rP": rP, "rF": rF, "kappa": kappa, "n_cambian": len(cambian), "n": len(m)}

def distribucion_por_linea(cT):
    """Cómo reparte el clasificador los productos entre líneas comerciales (alcance total)."""
    cls = cT["cls"].assign(valor=cT["sku"]["valor_12m"].values)
    t = cls.groupby(LINEA).agg(skus=("ItemCode", "size"), candidatos=("candidato", "sum"), pilotos=("piloto", "sum"),
                               P_medio=("P_consenso", "mean"), F_medio=("F_consenso", "mean"), valor=("valor", "sum"))
    t["pct_candidatos"] = t["candidatos"] / t["skus"] * 100; t["pct_valor"] = t["valor"] / t["valor"].sum() * 100
    t = t.drop(columns="valor").sort_values("pct_candidatos", ascending=False)
    display(t.round(2).reset_index())
    ct = pd.crosstab(cls[LINEA], cls["segmento"]).reindex(columns=SEGMENTOS, fill_value=0).loc[t.index]
    fig, ax = plt.subplots(figsize=(11, 0.9 * len(ct) + 1.8)); izq = np.zeros(len(ct))
    for s in SEGMENTOS:
        ax.barh(ct.index, ct[s], left=izq, color=SEG_COL[s], label=s, edgecolor="white")
        for i, (v, l) in enumerate(zip(ct[s], izq)):
            if v > 0: ax.text(l + v / 2, i, int(v), ha="center", va="center", fontsize=8, color="black")
        izq += ct[s].values
    ax.invert_yaxis(); ax.set_xlabel("Número de SKUs"); ax.set_title("Segmentos asignados por el clasificador, por línea comercial (universo total)", fontsize=10)
    ax.legend(fontsize=8, loc="lower right"); plt.tight_layout(); plt.show()
    return t

def texto_comparacion(cL, cT, tabla, cons, dist):
    a, b = tabla.loc["% candidatos"].values
    lin = LINEA_PRIORIZADA
    top = dist.sort_values("pct_candidatos", ascending=False)
    P = [f"**Del alcance por línea al universo total.** El mismo pipeline, sin cambios, produjo {int(tabla.loc['Candidatos'].iloc[0])} candidatos en {cL['nombre']} ({a:.0f} % de sus productos) "
         f"y {int(tabla.loc['Candidatos'].iloc[1])} en el {cT['nombre'].lower()} ({b:.0f} %). El número de clusters elegido fue {int(tabla.loc['k elegido'].iloc[0])} y {int(tabla.loc['k elegido'].iloc[1])}, "
         f"con un silhouette medio de {tabla.loc['Silhouette medio (k elegido)'].iloc[0]:.2f} y {tabla.loc['Silhouette medio (k elegido)'].iloc[1]:.2f}, respectivamente.",
         f"**Acuerdo entre métodos.** La W de Kendall en prioridad pasó de {tabla.loc['Kendall W prioridad'].iloc[0]:.2f} a {tabla.loc['Kendall W prioridad'].iloc[1]:.2f} y en pronosticabilidad "
         f"de {tabla.loc['Kendall W pronosticabilidad'].iloc[0]:.2f} a {tabla.loc['Kendall W pronosticabilidad'].iloc[1]:.2f}; el ARI medio entre métodos pasó de {tabla.loc['ARI medio entre métodos'].iloc[0]:.2f} a {tabla.loc['ARI medio entre métodos'].iloc[1]:.2f}. "
         "Con más productos el consenso entre métodos " + (
             "se fortalece, lo que sugiere que parte del desacuerdo observado en la línea se debía al tamaño reducido de la muestra." if tabla.loc["ARI medio entre métodos"].iloc[1] > tabla.loc["ARI medio entre métodos"].iloc[0] + .05
             else "se debilita, lo que sugiere que el desacuerdo no se explica por el tamaño reducido de la línea y aumenta con la heterogeneidad del portafolio." if tabla.loc["ARI medio entre métodos"].iloc[1] < tabla.loc["ARI medio entre métodos"].iloc[0] - .05
             else "se mantiene, lo que sugiere que el comportamiento del consenso no depende del tamaño del conjunto."),
         f"**Consistencia de los productos de {lin}.** Al clasificar sus {cons['n']} productos dentro del universo total, los percentiles se mantienen "
         f"{'muy parecidos' if min(cons['rP'], cons['rF']) > .8 else 'moderadamente parecidos' if min(cons['rP'], cons['rF']) > .5 else 'distintos'} "
         f"(Spearman de {cons['rP']:.2f} en prioridad y {cons['rF']:.2f} en pronosticabilidad), y el acuerdo en el estado de candidato fue {_nivel(cons['kappa'], [.2, .4, .6, .8], ['nulo o leve', 'discreto', 'moderado', 'sustancial', 'casi perfecto'])} "
         f"(kappa = {cons['kappa']:.2f}; {cons['n_cambian']} productos cambian de estado). Los percentiles cambian porque el ranking es relativo al conjunto: un producto que es alto dentro de su línea puede ser medio frente a todo el portafolio, y viceversa.",
         f"**Reparto por línea.** En el universo total, la línea con mayor proporción de candidatos fue {top.index[0]} ({top['pct_candidatos'].iloc[0]:.0f} %) y la menor {top.index[-1]} ({top['pct_candidatos'].iloc[-1]:.0f} %). "
         + (f"{lin} quedó con {dist.loc[lin, 'pct_candidatos']:.0f} % de candidatos, " + ("por encima" if dist.loc[lin, "pct_candidatos"] > b else "por debajo") + f" del promedio del portafolio ({b:.0f} %). " if lin in dist.index else "")
         + "Este reparto muestra qué líneas concentran los productos que reúnen a la vez alta prioridad y alta pronosticabilidad, y es el punto de partida para decidir si el piloto debe limitarse a una línea o ampliarse.",
         "**Lectura.** La clasificación sobre el total no sustituye a la de la línea priorizada: responde otra pregunta (qué productos del portafolio completo son mejores candidatos) y sirve para comprobar que el método no depende del tamaño del conjunto ni de las características de una sola línea."]
    display(Markdown("\n\n".join(P)))
    return "\n\n".join(P)

## Resultado 1 — Productos clasificados de la línea priorizada

Esta sección aplica el pipeline a los productos de la línea priorizada. Sus resultados alimentan la Fase 3.

In [ ]:
cL = nuevo_ctx("linea")
etapa_preparar(cL)
etapa_redundancia(cL)

### A2.2 (paso 3) — Elección del número de clusters

El número de clusters se elige entre 3 y 5 con el silhouette promedio de los tres algoritmos. No se incluye k = 2 porque separaría casi siempre "grandes" de "pequeños" y no distingue entre prioridad y pronosticabilidad. Los tres índices (silhouette, Davies-Bouldin y Calinski-Harabasz) se leen en conjunto: un silhouette por debajo de aproximadamente 0,25 indica que no hay grupos naturales marcados, y en ese caso el clustering aporta una partición útil pero no una estructura, y el peso de la decisión recae en los métodos multicriterio. La línea punteada de cada gráfica marca el k elegido; K-means y Ward suelen dar curvas casi superpuestas, por eso se dibujan con marcadores y trazos distintos.

In [ ]:
etapa_k(cL)

### A2.2 (paso 4) — Matriz producto × método

La tabla siguiente es el resultado central de la A2.2: una fila por producto, una columna por método con su voto (1 = propone al producto para el piloto), las clases de referencia ABC, XYZ y Syntetos-Boylan, los percentiles de consenso de los dos ejes y la proporción de votos. También se muestran los pesos que obtuvo cada criterio con cada esquema, que deben ir al anexo metodológico.

In [ ]:
etapa_matriz(cL)

## A4.1 — Evaluación de la clasificación

Como la segmentación no tiene etiquetas verdaderas, las métricas de exactitud no aplican. La evaluación se plantea en cuatro niveles: concordancia entre métodos, estabilidad, calidad interna de los clusters y validez externa frente al error de pronóstico.

| Pregunta | Métrica | Cómo leerla |
|---|---|---|
| ¿Los métodos ordenan igual? | Spearman entre puntajes y W de Kendall | W ≥ 0,7 indica acuerdo fuerte; valores bajos señalan que la selección depende del método. |
| ¿Deciden igual? | ARI y NMI entre votos | Cerca de 1, las decisiones coinciden; cerca de 0, son casi independientes. Si los métodos proponen grupos de tamaño muy distinto, el ARI es bajo aunque haya traslape. |
| ¿Los grupos son reales? | Silhouette, Davies-Bouldin, Calinski-Harabasz | Se interpretan en conjunto y con cautela cuando hay pocos productos. |
| ¿La selección es estable? | Frecuencia ante pesos aleatorios y ante submuestreo del 80 % | Un producto con frecuencia alta es robusto; uno cercano a 0,5 está en la frontera y es candidato a revisión manual. |
| ¿Sirve para pronosticar? | Spearman entre pronosticabilidad y MASE, y comparación piloto vs. resto | Se espera correlación negativa y menor MASE en el piloto (sección A2.3). |

La concordancia y las frecuencias de estabilidad sirven para sostener la selección ante el jurado, no para demostrar que es la correcta; esa afirmación solo la respalda la validez externa.

In [ ]:
etapa_concordancia(cL)

### Lectura de la concordancia en la línea priorizada

Las métricas de concordancia responden preguntas distintas y conviene leerlas por separado.

**Robustez del eje de pronosticabilidad.** Una W de Kendall alta en este eje significa que los nueve puntajes ordenan casi igual a los productos. Es esperable, porque sus cinco criterios (CV de 12 meses, meses atípicos, ADI, CV² e historia) miden esencialmente la misma idea de regularidad y varios están correlacionados entre sí. Que el orden no dependa del método ni de los pesos fortalece la selección, pero también indica que el eje aporta poca diversidad.

**Dependencia del eje de prioridad respecto de los pesos.** Sus criterios son casi independientes entre sí (devolución, vencimiento, cobertura y quiebres no se correlacionan mucho), de modo que la forma de resolver los conflictos entre criterios importa. En el mapa de calor suele verse que los esquemas de pesos objetivos (entropía y CRITIC) se parecen entre sí, que los pesos iguales por dimensión producen un orden distinto y que VIKOR, al penalizar el peor criterio de cada producto, es el método que más se separa de TOPSIS y PROMETHEE.

**Diferencias entre familias de métodos.** Un ARI alto dentro de una familia y bajo entre familias indica que cada familia identifica grupos distintos. Si las reglas clásicas (ABC·SB y ABC·XYZ) no coinciden con el multicriterio ni con el clustering, la lectura es metodológica: ABC y XYZ solo ven tamaño y regularidad, sin el costo del error en inventario, y por eso seleccionan productos distintos a los que se priorizan al combinar valor, riesgo y pronosticabilidad. De ahí la conveniencia del consenso por votos frente a cualquier método aislado. Debe tenerse en cuenta que el ARI sobre votos binarios penaliza los grupos de tamaño muy diferente: dos métodos pueden compartir productos y aun así mostrar un ARI bajo.

**Sobre el número de clusters.** Antes de aceptar el k elegido por silhouette promedio conviene revisar qué algoritmo lo impulsa y la columna `tam_min` de la tabla de validez, porque con pocos productos un k alto deja grupos pequeños. Para evaluar otro valor basta asignar, por ejemplo, `cL["k"] = 3` antes de correr la matriz y reportar la diferencia como análisis de sensibilidad.

In [ ]:
etapa_estabilidad(cL)

### A4.1 (parte 3) — Emulador supervisado como herramienta de explicación

Se entrena un árbol de decisión poco profundo para predecir si un producto es candidato al piloto. Su exactitud balanceada, medida con validación cruzada repetida (Raschka, 2018), indica **qué tan bien las reglas resultantes reproducen el consenso**, no si el consenso es correcto, porque la etiqueta se derivó de los mismos criterios. Su valor es práctico: el árbol traduce el consenso a reglas legibles ("valor alto y CV bajo") que el área comercial puede usar para clasificar productos nuevos sin repetir todo el análisis.

In [ ]:
etapa_emulador(cL)

## A2.3 — Selección y validación del grupo piloto

El piloto se arma con tres condiciones sucesivas: el producto debe ser candidato por consenso (proporción de votos mayor o igual a `VOTOS_MIN`), debe sobrevivir al submuestreo en una proporción mínima de las repeticiones (`FREC_MIN`) y, si hay más productos elegibles que el tope del alcance (`piloto_max`), se conservan los de mayor producto entre los percentiles de prioridad y de pronosticabilidad. En la línea priorizada el tope es de 20 productos y es una decisión de alcance, no estadística, que conviene fijar con el asesor pensando en cuántas series puede sostener el modelo de la Fase 3. En el universo total no se aplica tope, porque el objetivo es observar cómo actúa el clasificador.

Los productos que no entran al piloto no se descartan: se asignan a uno de tres segmentos según el cuadrante del plano prioridad × pronosticabilidad. Los prioritarios pero difíciles son candidatos a un método robusto para demanda intermitente (por ejemplo, Croston-SBA); los estables de bajo impacto, a un método simple; y los de bajo impacto y difíciles, solo a seguimiento. Esa asignación es lo que convierte la clasificación en un enfoque híbrido y no en un simple recorte del portafolio.

Después de la selección se muestra una tabla de sensibilidad a los umbrales: cuántos candidatos resultan al variar el corte por eje (`CORTE_EJE`) y la proporción mínima de votos (`VOTOS_MIN`). Es la forma de saber cuánto depende el tamaño del piloto de esas dos decisiones del investigador.

In [ ]:
etapa_seleccion(cL)
sensibilidad_umbrales(cL)

### A2.3 (parte 2) — Validez externa con un backtest sencillo

Para comprobar que el eje de pronosticabilidad ordena a los productos según el error real se corre un backtest de origen móvil con cuatro métodos simples (ingenuo, media móvil de 3 meses, suavización exponencial simple y Croston-SBA), sobre los últimos seis meses y un horizonte de un mes. El error se reporta como MASE, que compara contra el error de un pronóstico ingenuo dentro de la muestra y por eso es comparable entre productos de distinto volumen, y como WMAPE agregado, que pondera por volumen. La prueba no sustituye al modelo de la Fase 3, solo verifica que la clasificación tiene capacidad discriminante: si el piloto no tuviera un MASE menor que el resto, la segmentación no estaría aportando información sobre la dificultad de pronosticar. Se excluye el primer mes de cada producto por ser parcial. Los productos con demanda censurada por quiebres (más de `UMBRAL_CENSURA` % de días sin existencias) se reportan aparte, porque su error mide también la falta de inventario y no solo la dificultad de la serie.

In [ ]:
etapa_validacion(cL)

### Texto de apoyo para el análisis de resultados

La celda siguiente redacta el análisis con las cifras de esta ejecución (estructura de clusters, concordancia, selección, estabilidad, validez externa y cautelas). Se regenera si cambian los datos o los parámetros, así que puede copiarse a la tesis como borrador y ajustarse.

In [ ]:
texto_resultados(cL)

## A2.4 — Filtrado de la ABT y persistencia

Con el piloto definido se filtra la ABT para que la Fase 3 reciba únicamente sus series. Para cada alcance se guardan cuatro tablas en el Lakehouse: la matriz de clasificación completa con todos los votos y las métricas de estabilidad (trazabilidad para la tesis), la ABT de características del piloto, la ABT mensual del piloto (insumo del modelo de pronóstico) y la ABT mensual del resto de los productos, que sirve como base para el pronóstico simple de los no piloto. Las tablas del universo total llevan el sufijo `_total` y no deben usarse como insumo del modelo de la Fase 3: son el resultado del experimento.

In [ ]:
pil_L = etapa_guardar(cL)

## Resultado 2 — Productos clasificados del universo total

Esta sección ejecuta exactamente las mismas etapas con todos los productos del universo. Hay tres diferencias de contexto que explican por qué los números pueden cambiar sin que el método haya cambiado. Primero, los percentiles de consenso son relativos al conjunto, de modo que un producto "alto" ahora lo es frente a todo el portafolio y no solo frente a su línea. Segundo, la normalización y la imputación usan los rangos y las medianas del portafolio completo, donde conviven líneas con demanda muy estable y líneas muy irregulares. Tercero, con más productos los índices de clusters y las pruebas estadísticas tienen más potencia, y el submuestreo del 80 % es más estable. Como el objetivo es observar al clasificador, no se aplica tope de tamaño al piloto.

In [ ]:
cT = nuevo_ctx("total")
etapa_preparar(cT)
etapa_redundancia(cT)

In [ ]:
etapa_k(cT)

In [ ]:
etapa_matriz(cT)

In [ ]:
etapa_concordancia(cT)

In [ ]:
etapa_estabilidad(cT)

In [ ]:
etapa_emulador(cT)

In [ ]:
etapa_seleccion(cT)
sensibilidad_umbrales(cT)

In [ ]:
etapa_validacion(cT)

In [ ]:
texto_resultados(cT)

In [ ]:
pil_T = etapa_guardar(cT)

## Comparación de los dos resultados

Esta sección responde cuatro preguntas. ¿Cambia la calidad de la estructura y el acuerdo entre métodos al pasar de la línea priorizada al universo total? ¿Los productos de la línea priorizada reciben una clasificación parecida cuando se les clasifica solos y cuando se les clasifica dentro del total? ¿Cómo reparte el clasificador los productos entre las líneas comerciales? Y, por tanto, ¿el método depende del tamaño del conjunto? La consistencia se mide con la correlación de Spearman entre los percentiles de ambos alcances y con el coeficiente kappa de Cohen sobre el estado de candidato (que descuenta el acuerdo esperado por azar).

In [ ]:
tabla = comparar_alcances(cL, cT)
cons  = consistencia_linea_total(cL, cT)

In [ ]:
dist = distribucion_por_linea(cT)

In [ ]:
texto_cmp = texto_comparacion(cL, cT, tabla, cons, dist)

# Persistencia de la comparación y embudo de reducción para el texto de la tesis
guardar(tabla.reset_index().rename(columns={"index": "metrica"}).astype({"metrica": str}).round(4), "fs2_comparacion_alcances")
N_SKUS_CATALOGO = None   # SKUs del catálogo inicial (dato del entorno privado; None = no se muestra)
etapas = [("Catálogo inicial", N_SKUS_CATALOGO),
          ("Universo de la Fase 1", len(sku_all)),
          (f"Línea priorizada ({LINEA_PRIORIZADA})", len(cL["sku"])),
          ("Piloto de la línea", len(pil_L)),
          ("Candidatos del universo total", cT["sel"]["piloto"])]
embudo = pd.DataFrame([e for e in etapas if e[1] is not None], columns=["etapa", "skus"])
display(embudo)

In [ ]:
# ============================================================================
# Tablas de resumen para el documento (alcance de la línea priorizada)
# ============================================================================
prior = spark.table("fs2_priorizacion_lineas").toPandas()
display(prior[[LINEA, "skus", "topsis_dimension", "topsis_entropia",
               "pct_veces_primera", "rank_promedio", "viable"]].round(3))       # priorización de líneas (A2.1)

cls_L = cL["cls"]
display(cls_L[cls_L["candidato"]][["ItemCode", "ItemName", "abc", "xyz", "sb",
                                   "P_consenso", "F_consenso", "votos_piloto"]]
        .sort_values("votos_piloto", ascending=False))                        # candidatos al piloto

pares = cL["pares"]
print(pares if len(pares) else "Ningún par con |rho| >= 0,8")                 # redundancia entre criterios
display(pd.concat({"Prioridad": cL["r"]["WP"], "Pronosticabilidad": cL["r"]["WF"]})
        .round(3).reset_index())                                              # pesos por esquema (anexo)

### Limitaciones que deben quedar escritas en la tesis

La selección es una decisión de apoyo y depende de supuestos que conviene declarar: (1) con pocos productos en la línea, los índices de calidad de clusters y las pruebas de hipótesis tienen poca potencia, por lo que se complementan con estabilidad por submuestreo; (2) el eje de prioridad usa demanda pedida, que en una línea con devoluciones altas puede sobrestimar la demanda neta, y es una decisión pendiente si el modelo de la Fase 3 pronostica unidades pedidas o netas; (3) la demanda está censurada durante los quiebres y los indicadores de pronosticabilidad no la corrigen; (4) productos de una misma familia pueden sustituirse entre sí, lo que afecta la independencia de las series; (5) la priorización de la línea en la A2.1 se hizo con pocas alternativas, por lo que se apoyó en la simulación de pesos y no en la diferencia de puntajes; (6) los umbrales (`CORTE_EJE`, `VOTOS_MIN`, `FREC_MIN`, `piloto_max`) son decisiones del investigador, y por eso se reporta la sensibilidad del número de candidatos; y (7) la clasificación del universo total es un experimento de comportamiento del método y no sustituye a la clasificación de la línea priorizada.

### Referencias de apoyo

Syntetos, A. A., Boylan, J. E. y Croston, J. D. (2005), clasificación de la demanda; Kou, G., Lu, Y., Peng, Y. y Shi, Y. (2012), evaluación de clasificadores con métodos multicriterio; Strehl, A. y Ghosh, J. (2002), ensambles de clusters; Zietsman, D. y van Vuuren, J. H. (2022), segmentación de SKUs y pronóstico; Diakoulaki, D., Mavrotas, G. y Papayannakis, L. (1995), método CRITIC; Raschka, S. (2018), evaluación de modelos con muestras pequeñas. El detalle y las URL están en la tabla de literatura del proyecto.